# Yield Curve Theory and Practice

This notebook provides a comprehensive introduction to yield curves for quantitative finance practitioners. We'll cover both theoretical foundations and practical implementation using the FixedIncome package.

**Target Audience**: Junior quants with basic finance knowledge

**Learning Objectives**:
- Understand what yield curves are and why they matter
- Learn different types of yields and rates
- Master interpolation techniques
- Apply yield curves to bond pricing
- Analyze yield curve shapes and movements

## 1. Setup and Imports

First, let's import the necessary libraries and our FixedIncome package.

In [ ]:
import sys
sys.path.append('..')  # Add parent directory to access packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Import from FixedIncome package
from FixedIncome import YieldCurve, Bond
from FixedIncome.interpolation import (
    CubicSplineInterpolator,
    NelsonSiegelInterpolator,
    NelsonSiegelSvenssonInterpolator
)
from FixedIncome.data_fetchers import (
    fetch_us_treasury_data,
    parse_treasury_data_to_yield_curves
)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports successful!")

## 2. What is a Yield Curve?

A **yield curve** is a graph showing the relationship between the yield (interest rate) and time to maturity for bonds of similar credit quality. It's one of the most important tools in fixed income markets.

### Why Yield Curves Matter

1. **Pricing**: Discount cash flows to calculate bond prices
2. **Risk Management**: Measure interest rate exposure
3. **Trading**: Identify relative value opportunities
4. **Economic Indicator**: Predict economic conditions (e.g., recessions)

### Types of Yields

There are three main types of yields we'll encounter:

1. **Par Yield**: The coupon rate that makes a bond trade at par (face value)
2. **Spot Rate (Zero Rate)**: The yield on a zero-coupon bond
3. **Forward Rate**: The implied future interest rate between two points in time

**Key Relationship**: Forward rates are derived from spot rates, and par yields are derived from spot rates.

## 3. Loading Real Market Data

Let's fetch current US Treasury yield curve data from the US Department of Treasury website. This data is freely available and updated daily.

In [ ]:
# Fetch current year US Treasury data
current_year = datetime.now().year
print(f"Fetching US Treasury data for {current_year}...")

treasury_df = fetch_us_treasury_data(year=current_year)

print(f"\nData shape: {treasury_df.shape}")
print(f"Date range: {treasury_df['Date'].min()} to {treasury_df['Date'].max()}")
print(f"\nAvailable maturities: {[col for col in treasury_df.columns if col != 'Date']}")

# Display first few rows
treasury_df.head()

### Understanding the Data

The US Treasury publishes **par yields** for constant maturity treasuries. These are:
- **1 Mo, 3 Mo, 6 Mo**: Short-term rates (money market)
- **1 Yr, 2 Yr, 3 Yr, 5 Yr, 7 Yr**: Medium-term rates
- **10 Yr, 20 Yr, 30 Yr**: Long-term rates

Note: Missing values (NaN) can occur when certain maturities aren't traded on a given day.

In [ ]:
# Get the most recent date with complete data
recent_data = treasury_df.dropna().tail(1)

if len(recent_data) == 0:
    print("No complete data found, using most recent row with partial data")
    recent_data = treasury_df.tail(1)

print(f"Using data from: {recent_data['Date'].values[0]}")
print("\nYields by maturity:")
print(recent_data.iloc[0].drop('Date'))

## 4. Creating a YieldCurve Object

Now let's use our FixedIncome package to create a YieldCurve object. This provides a clean interface for working with yield data.

In [ ]:
# Extract tenors and yields from the data
# Convert maturity labels to years (tenor)
maturity_map = {
    '1 Mo': 1/12,
    '3 Mo': 0.25,
    '6 Mo': 0.5,
    '1 Yr': 1.0,
    '2 Yr': 2.0,
    '3 Yr': 3.0,
    '5 Yr': 5.0,
    '7 Yr': 7.0,
    '10 Yr': 10.0,
    '20 Yr': 20.0,
    '30 Yr': 30.0
}

# Extract data for the most recent date
tenors = []
yields = []

for col, tenor in maturity_map.items():
    if col in recent_data.columns:
        yield_value = recent_data[col].values[0]
        if not pd.isna(yield_value):
            tenors.append(tenor)
            yields.append(yield_value / 100)  # Convert from percentage to decimal

print(f"Loaded {len(tenors)} points")
print(f"Tenor range: {min(tenors):.2f} to {max(tenors):.1f} years")

# Create YieldCurve object
yc = YieldCurve(
    tenors=tenors,
    yields=yields,
    date=pd.to_datetime(recent_data['Date'].values[0]),
    currency='USD'
)

print(f"\nYieldCurve created: {yc}")

### Basic Yield Curve Visualization

In [ ]:
# Plot the raw data points
plt.figure(figsize=(12, 6))
plt.plot(yc.tenors, yc.yields * 100, 'o-', markersize=8, linewidth=2, label='Market Data')
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title(f'US Treasury Yield Curve - {yc.date.strftime("%Y-%m-%d")}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Current curve shape: {yc.get_shape_metrics()}")

## 5. Yield Curve Shapes and Economic Interpretation

The shape of the yield curve provides important economic information:

### Common Shapes

1. **Normal (Upward Sloping)**: Long-term yields > Short-term yields
   - Indicates economic expansion expected
   - Compensation for time and inflation risk

2. **Inverted (Downward Sloping)**: Short-term yields > Long-term yields
   - Often precedes recessions
   - Market expects rate cuts in the future

3. **Flat**: Similar yields across maturities
   - Transition period
   - Uncertainty about future direction

4. **Humped**: Peak in medium term
   - Unusual shape
   - Can indicate policy expectations

### Shape Metrics

The YieldCurve class provides quantitative shape metrics:

In [ ]:
# Calculate shape metrics
metrics = yc.get_shape_metrics()

print("Yield Curve Shape Analysis")
print("=" * 50)
print(f"Level (10Y yield):     {metrics['level']:.4f} ({metrics['level']*100:.2f}%)")
print(f"Slope (10Y - 2Y):      {metrics['slope_2_10']:.4f} ({metrics['slope_2_10']*100:.2f} bps)")
print(f"Curvature (Butterfly): {metrics['curvature']:.4f} ({metrics['curvature']*100:.2f} bps)")
print()

# Interpretation
if metrics['slope_2_10'] > 0:
    print("Slope: POSITIVE (Normal curve) - Economic expansion expected")
elif metrics['slope_2_10'] < 0:
    print("Slope: NEGATIVE (Inverted curve) - Potential recession signal")
else:
    print("Slope: FLAT - Transition period")

if abs(metrics['curvature']) > 0.001:
    print(f"Curvature: {'POSITIVE' if metrics['curvature'] > 0 else 'NEGATIVE'} (Pronounced hump)")
else:
    print("Curvature: FLAT (Linear curve)")

## 6. Interpolation: Filling the Gaps

Market data gives us yields at specific maturities (1Y, 2Y, 5Y, etc.), but we often need yields at intermediate points (e.g., 3.5 years). This is where **interpolation** comes in.

### Why Interpolation Matters

- Price bonds with arbitrary maturities
- Calculate risk metrics at any point on the curve
- Create smooth curves for visualization and analysis

### Methods We'll Explore

1. **Cubic Spline**: Smooth piecewise polynomials
2. **Nelson-Siegel**: Parametric model with 4 parameters
3. **Nelson-Siegel-Svensson**: Extended NS with 6 parameters

Let's try each method on our yield curve data.

### 6.1 Cubic Spline Interpolation

Cubic splines create smooth curves by fitting piecewise cubic polynomials between data points.

**Pros**: Smooth, passes through all data points exactly

**Cons**: Can create unrealistic wiggles, no economic intuition

In [ ]:
# Create and fit cubic spline interpolator
cs_interpolator = CubicSplineInterpolator(bc_type='natural')
cs_interpolator.fit(np.array(yc.tenors), np.array(yc.yields))

# Set it as the YieldCurve's interpolator
yc.set_interpolator(cs_interpolator)

# Generate interpolated points
fine_tenors = np.linspace(min(yc.tenors), max(yc.tenors), 100)
cs_yields = np.array([yc.get_yield(t) for t in fine_tenors])

# Plot
plt.figure(figsize=(12, 6))
plt.plot(yc.tenors, yc.yields * 100, 'o', markersize=10, label='Market Data', color='blue')
plt.plot(fine_tenors, cs_yields * 100, '-', linewidth=2, label='Cubic Spline', color='red')
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title('Cubic Spline Interpolation', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Example: Get yield at 3.5 years
y_3_5 = yc.get_yield(3.5)
print(f"\nInterpolated yield at 3.5 years: {y_3_5:.4f} ({y_3_5*100:.2f}%)")

### 6.2 Nelson-Siegel Model

The Nelson-Siegel model is a parametric approach that models the entire yield curve with 4 parameters:

$$y(t) = \beta_0 + \beta_1 \cdot \frac{1 - e^{-t/\tau}}{t/\tau} + \beta_2 \cdot \left(\frac{1 - e^{-t/\tau}}{t/\tau} - e^{-t/\tau}\right)$$

**Parameters**:
- $\beta_0$: Long-term level (as $t \to \infty$, $y(t) \to \beta_0$)
- $\beta_1$: Short-term component (slope at origin)
- $\beta_2$: Medium-term component (curvature)
- $\tau$: Decay parameter (controls location of maximum curvature)

**Pros**: Economic interpretation, smooth, good for forecasting

**Cons**: May not fit data perfectly, 4 parameters may be limiting

In [ ]:
# Create and fit Nelson-Siegel interpolator
ns_interpolator = NelsonSiegelInterpolator()
ns_interpolator.fit(np.array(yc.tenors), np.array(yc.yields))

# Get fitted parameters
ns_params = ns_interpolator.get_parameters()
print("Nelson-Siegel Parameters:")
print("=" * 50)
for param, value in ns_params.items():
    print(f"{param:8s} = {value:8.4f}")

# Generate interpolated curve
ns_yields = np.array([ns_interpolator.interpolate(t) for t in fine_tenors])

# Plot comparison
plt.figure(figsize=(12, 6))
plt.plot(yc.tenors, yc.yields * 100, 'o', markersize=10, label='Market Data', color='blue')
plt.plot(fine_tenors, ns_yields * 100, '-', linewidth=2, label='Nelson-Siegel', color='green')
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title('Nelson-Siegel Interpolation', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate fit quality (RMSE)
ns_fitted = np.array([ns_interpolator.interpolate(t) for t in yc.tenors])
rmse = np.sqrt(np.mean((ns_fitted - yc.yields)**2))
print(f"\nRoot Mean Square Error: {rmse:.6f} ({rmse*10000:.2f} bps)")

### 6.3 Nelson-Siegel-Svensson Model

The NSS model extends Nelson-Siegel with an additional term for more flexibility:

$$y(t) = \beta_0 + \beta_1 \cdot \frac{1 - e^{-t/\tau_1}}{t/\tau_1} + \beta_2 \cdot \left(\frac{1 - e^{-t/\tau_1}}{t/\tau_1} - e^{-t/\tau_1}\right) + \beta_3 \cdot \left(\frac{1 - e^{-t/\tau_2}}{t/\tau_2} - e^{-t/\tau_2}\right)$$

**Additional Parameters**:
- $\beta_3$: Second curvature component
- $\tau_2$: Second decay parameter

**Pros**: More flexible, can capture complex shapes

**Cons**: More parameters = risk of overfitting, slower to fit

In [ ]:
# Create and fit NSS interpolator
nss_interpolator = NelsonSiegelSvenssonInterpolator()
nss_interpolator.fit(np.array(yc.tenors), np.array(yc.yields))

# Get fitted parameters
nss_params = nss_interpolator.get_parameters()
print("Nelson-Siegel-Svensson Parameters:")
print("=" * 50)
for param, value in nss_params.items():
    print(f"{param:8s} = {value:8.4f}")

# Generate interpolated curve
nss_yields = np.array([nss_interpolator.interpolate(t) for t in fine_tenors])

# Plot comparison
plt.figure(figsize=(12, 6))
plt.plot(yc.tenors, yc.yields * 100, 'o', markersize=10, label='Market Data', color='blue')
plt.plot(fine_tenors, nss_yields * 100, '-', linewidth=2, label='Nelson-Siegel-Svensson', color='purple')
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title('Nelson-Siegel-Svensson Interpolation', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate fit quality
nss_fitted = np.array([nss_interpolator.interpolate(t) for t in yc.tenors])
rmse = np.sqrt(np.mean((nss_fitted - yc.yields)**2))
print(f"\nRoot Mean Square Error: {rmse:.6f} ({rmse*10000:.2f} bps)")

### 6.4 Comparing All Methods

In [ ]:
# Plot all methods together
plt.figure(figsize=(14, 7))
plt.plot(yc.tenors, yc.yields * 100, 'o', markersize=10, label='Market Data', color='blue', zorder=5)
plt.plot(fine_tenors, cs_yields * 100, '-', linewidth=2, label='Cubic Spline', alpha=0.7)
plt.plot(fine_tenors, ns_yields * 100, '--', linewidth=2, label='Nelson-Siegel', alpha=0.7)
plt.plot(fine_tenors, nss_yields * 100, '-.', linewidth=2, label='Nelson-Siegel-Svensson', alpha=0.7)
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title('Comparison of Interpolation Methods', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate and compare RMSE for all methods
print("\nFit Quality Comparison (RMSE in basis points):")
print("=" * 50)
cs_fitted = np.array([cs_interpolator.interpolate(t) for t in yc.tenors])
cs_rmse = np.sqrt(np.mean((cs_fitted - yc.yields)**2)) * 10000
ns_rmse = np.sqrt(np.mean((ns_fitted - yc.yields)**2)) * 10000
nss_rmse = np.sqrt(np.mean((nss_fitted - yc.yields)**2)) * 10000

print(f"Cubic Spline:            {cs_rmse:.2f} bps")
print(f"Nelson-Siegel:           {ns_rmse:.2f} bps")
print(f"Nelson-Siegel-Svensson:  {nss_rmse:.2f} bps")

## 7. Spot Rates, Forward Rates, and Discount Factors

Now let's explore the key rate concepts and how they relate to each other.

### 7.1 Discount Factors

A **discount factor** $DF(t)$ tells us the present value of $1 received at time $t$:

$$DF(t) = e^{-y(t) \cdot t}$$

where $y(t)$ is the continuously compounded spot rate.

Discount factors are fundamental to pricing: any cash flow $C$ at time $t$ has present value $PV = C \times DF(t)$.

In [ ]:
# Calculate discount factors at various points
test_tenors = [0.5, 1, 2, 5, 10, 20, 30]

print("Discount Factors and Spot Rates")
print("=" * 60)
print(f"{'Maturity':>10} | {'Spot Rate':>12} | {'Discount Factor':>15}")
print("-" * 60)

for t in test_tenors:
    if t <= max(yc.tenors):
        spot_rate = yc.get_yield(t)
        df = yc.get_discount_factor(t)
        print(f"{t:>8.1f}Y | {spot_rate*100:>10.3f}% | {df:>15.6f}")

# Visualize discount factors
df_tenors = np.linspace(min(yc.tenors), max(yc.tenors), 100)
discount_factors = [yc.get_discount_factor(t) for t in df_tenors]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot yield curve
ax1.plot(fine_tenors, cs_yields * 100, linewidth=2, color='blue')
ax1.set_xlabel('Maturity (Years)', fontsize=11)
ax1.set_ylabel('Yield (%)', fontsize=11)
ax1.set_title('Spot Rate Curve', fontsize=12)
ax1.grid(True, alpha=0.3)

# Plot discount factor curve
ax2.plot(df_tenors, discount_factors, linewidth=2, color='red')
ax2.set_xlabel('Maturity (Years)', fontsize=11)
ax2.set_ylabel('Discount Factor', fontsize=11)
ax2.set_title('Discount Factor Curve', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2 Forward Rates

A **forward rate** $f(t_1, t_2)$ is the implied interest rate for borrowing/lending between times $t_1$ and $t_2$.

**Formula**:
$$f(t_1, t_2) = \frac{y(t_2) \cdot t_2 - y(t_1) \cdot t_1}{t_2 - t_1}$$

**Key Insight**: Forward rates tell us what the market expects interest rates to be in the future. This is crucial for:
- Trading strategies (riding the yield curve)
- Hedging future borrowing costs
- Understanding market expectations

**No-arbitrage relationship**: Investing for period $(0, t_2)$ should give the same return as investing for $(0, t_1)$ and then reinvesting at the forward rate $f(t_1, t_2)$ for period $(t_1, t_2)$.

In [ ]:
# Calculate forward rates
print("Forward Rates Analysis")
print("=" * 70)
print(f"{'Period':>15} | {'Spot 1':>10} | {'Spot 2':>10} | {'Forward':>10}")
print("-" * 70)

forward_periods = [(1, 2), (2, 5), (5, 10), (10, 20)]

for t1, t2 in forward_periods:
    if t2 <= max(yc.tenors):
        spot1 = yc.get_yield(t1)
        spot2 = yc.get_yield(t2)
        forward = yc.get_forward_rate(t1, t2)
        print(f"{t1}Y → {t2}Y | {spot1*100:>8.3f}% | {spot2*100:>8.3f}% | {forward*100:>8.3f}%")

# Visualize instantaneous forward rate curve
# Instantaneous forward: f(t) = y(t) + t * dy/dt
dt = 0.01
inst_forward_tenors = np.arange(min(yc.tenors) + dt, max(yc.tenors), 0.1)
inst_forward_rates = []

for t in inst_forward_tenors:
    # Numerical derivative
    y1 = yc.get_yield(t - dt/2)
    y2 = yc.get_yield(t + dt/2)
    dy_dt = (y2 - y1) / dt
    y_t = yc.get_yield(t)
    inst_forward = y_t + t * dy_dt
    inst_forward_rates.append(inst_forward)

# Plot spot vs forward
plt.figure(figsize=(12, 6))
plt.plot(fine_tenors, cs_yields * 100, linewidth=2, label='Spot Rates', color='blue')
plt.plot(inst_forward_tenors, np.array(inst_forward_rates) * 100, linewidth=2, 
         label='Instantaneous Forward Rates', color='red', linestyle='--')
plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Rate (%)', fontsize=12)
plt.title('Spot Rates vs Forward Rates', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey Observation:")
print("When the curve is upward sloping, forward rates > spot rates")
print("This implies the market expects rates to rise over time.")

## 8. Bond Pricing with Yield Curves

Now let's apply our yield curve to price bonds. This connects theory to practice.

### Bond Pricing Formula

A bond's price is the present value of its cash flows:

$$P = \sum_{i=1}^{n} C_i \times DF(t_i) = \sum_{i=1}^{n} C_i \times e^{-y(t_i) \cdot t_i}$$

where:
- $C_i$ = cash flow at time $t_i$ (coupon or principal)
- $DF(t_i)$ = discount factor at time $t_i$
- $y(t_i)$ = spot rate at time $t_i$

In [ ]:
# Create sample bonds with different characteristics
bonds = [
    Bond(face_value=1000, coupon_rate=0.03, maturity=2, frequency=2),
    Bond(face_value=1000, coupon_rate=0.04, maturity=5, frequency=2),
    Bond(face_value=1000, coupon_rate=0.05, maturity=10, frequency=2),
]

print("Bond Pricing Using Yield Curve")
print("=" * 80)
print(f"{'Maturity':>10} | {'Coupon':>8} | {'Price':>12} | {'YTM':>8} | {'Duration':>10}")
print("-" * 80)

for bond in bonds:
    # Get cash flows
    cash_flows = bond.get_cash_flows()
    
    # Price using yield curve (discount each cash flow separately)
    price = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows)
    
    # Calculate YTM
    ytm = bond.yield_to_maturity(price)
    
    # Calculate duration (approximate)
    duration = bond.duration(ytm if ytm else 0.05)
    
    print(f"{bond.maturity:>8.1f}Y | {bond.coupon_rate*100:>6.1f}% | ${price:>10.2f} | {(ytm*100 if ytm else 0):>6.2f}% | {duration:>8.2f}")

### Understanding the Results

**Price vs Par**: 
- If Price > $1000: Bond trades at a premium (coupon > market yield)
- If Price < $1000: Bond trades at a discount (coupon < market yield)
- If Price = $1000: Bond trades at par (coupon = market yield)

**YTM vs Spot Rates**:
- YTM is a single rate that discounts all cash flows to the current price
- Spot rates vary by maturity
- YTM is approximately a weighted average of the relevant spot rates

## 9. Duration and Convexity from Yield Curves

**Duration** measures a bond's sensitivity to interest rate changes. It's essential for risk management.

### Macaulay Duration

$$D = \frac{1}{P} \sum_{i=1}^{n} t_i \cdot C_i \cdot DF(t_i)$$

**Interpretation**: The weighted average time to receive cash flows

### Modified Duration

$$D_{\text{mod}} = \frac{D}{1 + y/m}$$

where $m$ is the compounding frequency.

**Interpretation**: Approximate percentage price change for a 1% change in yield:
$$\Delta P \approx -D_{\text{mod}} \times \Delta y \times P$$

### Convexity

Convexity measures the curvature of the price-yield relationship. It improves duration-based estimates:

$$\Delta P \approx -D_{\text{mod}} \times \Delta y \times P + \frac{1}{2} C \times (\Delta y)^2 \times P$$

In [ ]:
# Detailed analysis for a 10-year bond
bond_10y = Bond(face_value=1000, coupon_rate=0.05, maturity=10, frequency=2)
cash_flows = bond_10y.get_cash_flows()

# Price using yield curve
price = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows)
ytm = bond_10y.yield_to_maturity(price)

print("10-Year Bond Risk Metrics")
print("=" * 50)
print(f"Coupon Rate:       {bond_10y.coupon_rate*100:.2f}%")
print(f"Current Price:     ${price:.2f}")
print(f"Yield to Maturity: {(ytm*100 if ytm else 0):.3f}%")
print()

if ytm:
    duration = bond_10y.duration(ytm)
    mod_duration = bond_10y.modified_duration(ytm)
    convexity = bond_10y.convexity(ytm)
    
    print(f"Macaulay Duration:   {duration:.3f} years")
    print(f"Modified Duration:   {mod_duration:.3f}")
    print(f"Convexity:          {convexity:.2f}")
    print()
    
    # Simulate rate changes
    rate_changes_bps = [-100, -50, -25, 0, 25, 50, 100]
    
    print("Impact of Rate Changes:")
    print("-" * 70)
    print(f"{'Rate Change':>12} | {'New Price':>12} | {'Duration Est':>14} | {'Duration+Convexity':>18}")
    print("-" * 70)
    
    for bps in rate_changes_bps:
        dy = bps / 10000  # Convert bps to decimal
        new_ytm = ytm + dy
        
        # Actual new price
        new_price_actual = bond_10y.price(new_ytm)
        
        # Duration approximation
        duration_est = price * (1 - mod_duration * dy)
        
        # Duration + Convexity approximation
        duration_convexity_est = price * (1 - mod_duration * dy + 0.5 * convexity * dy**2)
        
        print(f"{bps:>10} bps | ${new_price_actual:>10.2f} | ${duration_est:>12.2f} | ${duration_convexity_est:>16.2f}")
    
    print("\nKey Insight: Convexity improves the estimate, especially for large rate changes.")

## 10. Yield Curve Shifts and Scenarios

In risk management, we analyze how bond portfolios respond to different yield curve shifts:

1. **Parallel Shift**: All rates move by the same amount
2. **Steepening**: Long rates rise more than short rates (slope increases)
3. **Flattening**: Short rates rise more than long rates (slope decreases)
4. **Twist**: Short and long rates move in opposite directions

In [ ]:
# Create scenario curves
shift_bps = 50  # 50 basis points

# Base curve
base_yields = np.array([yc.get_yield(t) for t in fine_tenors])

# Parallel shift: add same amount to all rates
parallel_up = base_yields + shift_bps / 10000
parallel_down = base_yields - shift_bps / 10000

# Steepening: long rates up, short rates unchanged
steepening_shift = np.linspace(0, shift_bps / 10000, len(fine_tenors))
steepening = base_yields + steepening_shift

# Flattening: short rates up, long rates unchanged
flattening_shift = np.linspace(shift_bps / 10000, 0, len(fine_tenors))
flattening = base_yields + flattening_shift

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Parallel shifts
axes[0, 0].plot(fine_tenors, base_yields * 100, 'k-', linewidth=2, label='Base')
axes[0, 0].plot(fine_tenors, parallel_up * 100, 'r--', linewidth=2, label=f'+{shift_bps} bps')
axes[0, 0].plot(fine_tenors, parallel_down * 100, 'g--', linewidth=2, label=f'-{shift_bps} bps')
axes[0, 0].set_title('Parallel Shifts', fontsize=12)
axes[0, 0].set_xlabel('Maturity (Years)')
axes[0, 0].set_ylabel('Yield (%)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Steepening
axes[0, 1].plot(fine_tenors, base_yields * 100, 'k-', linewidth=2, label='Base')
axes[0, 1].plot(fine_tenors, steepening * 100, 'b--', linewidth=2, label='Steepening')
axes[0, 1].set_title('Steepening', fontsize=12)
axes[0, 1].set_xlabel('Maturity (Years)')
axes[0, 1].set_ylabel('Yield (%)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Flattening
axes[1, 0].plot(fine_tenors, base_yields * 100, 'k-', linewidth=2, label='Base')
axes[1, 0].plot(fine_tenors, flattening * 100, 'm--', linewidth=2, label='Flattening')
axes[1, 0].set_title('Flattening', fontsize=12)
axes[1, 0].set_xlabel('Maturity (Years)')
axes[1, 0].set_ylabel('Yield (%)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# All scenarios together
axes[1, 1].plot(fine_tenors, base_yields * 100, 'k-', linewidth=2.5, label='Base')
axes[1, 1].plot(fine_tenors, parallel_up * 100, 'r:', linewidth=1.5, alpha=0.7, label='Parallel Up')
axes[1, 1].plot(fine_tenors, steepening * 100, 'b:', linewidth=1.5, alpha=0.7, label='Steepening')
axes[1, 1].plot(fine_tenors, flattening * 100, 'm:', linewidth=1.5, alpha=0.7, label='Flattening')
axes[1, 1].set_title('All Scenarios', fontsize=12)
axes[1, 1].set_xlabel('Maturity (Years)')
axes[1, 1].set_ylabel('Yield (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Portfolio Impact Analysis

Let's see how a bond portfolio responds to these scenarios.

In [ ]:
# Create a simple portfolio
portfolio = [
    {'bond': Bond(1000, 0.03, 2, 2), 'quantity': 100},   # Short duration
    {'bond': Bond(1000, 0.04, 5, 2), 'quantity': 50},    # Medium duration
    {'bond': Bond(1000, 0.05, 10, 2), 'quantity': 25},   # Long duration
]

# Calculate base portfolio value
base_value = 0
for position in portfolio:
    bond = position['bond']
    qty = position['quantity']
    cash_flows = bond.get_cash_flows()
    price = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows)
    base_value += price * qty

print(f"Base Portfolio Value: ${base_value:,.2f}")
print()

# Function to calculate portfolio value with shifted curve
def portfolio_value_with_shift(yields_shifted):
    # Create temporary yield curve with shifted yields
    yc_shifted = YieldCurve(tenors=yc.tenors, yields=yc.yields)
    # Set interpolator with shifted data
    cs_shifted = CubicSplineInterpolator()
    shifted_yields_at_tenors = np.interp(yc.tenors, fine_tenors, yields_shifted)
    cs_shifted.fit(np.array(yc.tenors), shifted_yields_at_tenors)
    yc_shifted.set_interpolator(cs_shifted)
    
    # Revalue portfolio
    total = 0
    for position in portfolio:
        bond = position['bond']
        qty = position['quantity']
        cash_flows = bond.get_cash_flows()
        price = sum(cf.amount * yc_shifted.get_discount_factor(cf.time) for cf in cash_flows)
        total += price * qty
    return total

# Calculate P&L for each scenario
scenarios = {
    'Parallel Up (+50bp)': parallel_up,
    'Parallel Down (-50bp)': parallel_down,
    'Steepening': steepening,
    'Flattening': flattening,
}

print("Portfolio P&L Under Different Scenarios")
print("=" * 60)
print(f"{'Scenario':>25} | {'New Value':>15} | {'P&L':>12} | {'P&L %':>8}")
print("-" * 60)

for scenario_name, shifted_yields in scenarios.items():
    new_value = portfolio_value_with_shift(shifted_yields)
    pnl = new_value - base_value
    pnl_pct = (pnl / base_value) * 100
    
    print(f"{scenario_name:>25} | ${new_value:>13,.2f} | ${pnl:>10,.2f} | {pnl_pct:>6.2f}%")

print("\nKey Observations:")
print("- Parallel up: Portfolio loses value (rates up, prices down)")
print("- Parallel down: Portfolio gains value (rates down, prices up)")
print("- Steepening/Flattening: Impact depends on portfolio's duration distribution")

## 11. Historical Yield Curve Analysis

Let's fetch multiple dates of yield curve data to see how the curve has evolved over time.

In [ ]:
# Fetch data and parse to YieldCurve objects
print("Fetching historical yield curve data...")
yield_curves_dict = parse_treasury_data_to_yield_curves(
    treasury_df,
    sample_frequency='weekly',  # Sample weekly to reduce data points
    maturity_columns=None  # Use all available maturities
)

print(f"Loaded {len(yield_curves_dict)} yield curve snapshots")
print(f"Date range: {min(yield_curves_dict.keys())} to {max(yield_curves_dict.keys())}")

# Plot evolution
plt.figure(figsize=(14, 7))

# Sample a few dates for clarity
dates_to_plot = sorted(yield_curves_dict.keys())
step = max(len(dates_to_plot) // 6, 1)  # Plot ~6 curves
dates_sample = dates_to_plot[::step]

for date in dates_sample:
    yc_hist = yield_curves_dict[date]
    plt.plot(yc_hist.tenors, np.array(yc_hist.yields) * 100, 
             marker='o', markersize=4, linewidth=1.5, 
             label=date.strftime('%Y-%m-%d'), alpha=0.7)

plt.xlabel('Maturity (Years)', fontsize=12)
plt.ylabel('Yield (%)', fontsize=12)
plt.title(f'Historical Yield Curves - {current_year}', fontsize=14)
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Analyzing Yield Curve Movements

Let's track how key metrics have changed over time.

In [ ]:
# Extract metrics over time
dates = []
levels = []
slopes = []
curvatures = []

for date in sorted(yield_curves_dict.keys()):
    yc_hist = yield_curves_dict[date]
    metrics = yc_hist.get_shape_metrics()
    
    dates.append(date)
    levels.append(metrics['level'] * 100)
    slopes.append(metrics['slope_2_10'] * 100)
    curvatures.append(metrics['curvature'] * 100)

# Create time series plots
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Level (10Y yield)
axes[0].plot(dates, levels, linewidth=2, color='blue')
axes[0].set_ylabel('Level - 10Y Yield (%)', fontsize=11)
axes[0].set_title('Yield Curve Metrics Over Time', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Slope (10Y - 2Y)
axes[1].plot(dates, slopes, linewidth=2, color='green')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_ylabel('Slope - 10Y-2Y (bps)', fontsize=11)
axes[1].grid(True, alpha=0.3)

# Curvature
axes[2].plot(dates, curvatures, linewidth=2, color='purple')
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[2].set_ylabel('Curvature (bps)', fontsize=11)
axes[2].set_xlabel('Date', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nYield Curve Metrics Summary")
print("=" * 50)
print(f"{'Metric':>15} | {'Current':>10} | {'Min':>10} | {'Max':>10} | {'Mean':>10}")
print("-" * 50)
print(f"{'Level (%)':>15} | {levels[-1]:>10.2f} | {min(levels):>10.2f} | {max(levels):>10.2f} | {np.mean(levels):>10.2f}")
print(f"{'Slope (bps)':>15} | {slopes[-1]:>10.0f} | {min(slopes):>10.0f} | {max(slopes):>10.0f} | {np.mean(slopes):>10.0f}")
print(f"{'Curvature (bps)':>15} | {curvatures[-1]:>10.0f} | {min(curvatures):>10.0f} | {max(curvatures):>10.0f} | {np.mean(curvatures):>10.0f}")

## 12. Practical Applications and Trading Strategies

Now that we understand yield curves, let's explore practical applications.

### 12.1 Riding the Yield Curve

**Strategy**: Buy bonds with maturity longer than your investment horizon and sell before maturity.

**Profit mechanism**: If the yield curve is upward sloping and stable:
1. You earn higher coupon on longer-maturity bonds
2. As time passes, the bond "rolls down" the curve to lower yields
3. Lower yield = higher price = capital gain

**Risk**: Curve steepens (long rates rise), causing losses

In [ ]:
# Simulate roll-down return
print("Riding the Yield Curve - Scenario Analysis")
print("=" * 70)

# Strategy: Buy 5Y bond, hold for 1 year (becomes 4Y bond)
initial_maturity = 5.0
holding_period = 1.0
final_maturity = initial_maturity - holding_period

# Current yields
y_5y = yc.get_yield(initial_maturity)
y_4y = yc.get_yield(final_maturity)

print(f"Initial position: Buy 5Y bond")
print(f"5Y yield today:   {y_5y*100:.3f}%")
print(f"4Y yield today:   {y_4y*100:.3f}%")
print(f"Holding period:   {holding_period:.1f} year")
print()

# Create a 5Y par bond
bond = Bond(face_value=1000, coupon_rate=y_5y, maturity=initial_maturity, frequency=2)

# Initial price (should be ~par)
initial_price = bond.price(y_5y)
print(f"Purchase price: ${initial_price:.2f}")
print()

# Scenario 1: Curve unchanged (roll-down gains)
print("Scenario 1: Curve Unchanged (Roll-Down)")
# After 1 year, bond has 4Y to maturity, price at 4Y yield
future_price_rolldown = bond.price(y_4y)  # This uses 4Y yield for remaining 4Y
coupon_income = y_5y * 1000  # Approximate annual coupon
total_return_rolldown = (future_price_rolldown - initial_price + coupon_income) / initial_price
print(f"  Price after 1Y: ${future_price_rolldown:.2f}")
print(f"  Coupon income:  ${coupon_income:.2f}")
print(f"  Total return:   {total_return_rolldown*100:.2f}%")
print()

# Scenario 2: Parallel shift up (+50bp)
print("Scenario 2: Rates Up 50bp (Parallel)")
y_4y_up = y_4y + 0.005
future_price_up = bond.price(y_4y_up)
total_return_up = (future_price_up - initial_price + coupon_income) / initial_price
print(f"  Price after 1Y: ${future_price_up:.2f}")
print(f"  Coupon income:  ${coupon_income:.2f}")
print(f"  Total return:   {total_return_up*100:.2f}%")
print()

# Scenario 3: Parallel shift down (-50bp)
print("Scenario 3: Rates Down 50bp (Parallel)")
y_4y_down = y_4y - 0.005
future_price_down = bond.price(y_4y_down)
total_return_down = (future_price_down - initial_price + coupon_income) / initial_price
print(f"  Price after 1Y: ${future_price_down:.2f}")
print(f"  Coupon income:  ${coupon_income:.2f}")
print(f"  Total return:   {total_return_down*100:.2f}%")
print()

print("Key Insight: Roll-down strategy profits when curve is stable or steepens,")
print("             but loses when rates rise significantly.")

### 12.2 Curve Positioning: Bullet vs Barbell

Two common strategies with similar duration but different convexity:

**Bullet**: Concentrate positions in one maturity (e.g., all 10Y bonds)

**Barbell**: Split between short and long maturities (e.g., 2Y and 30Y bonds)

**Key difference**: Barbell has higher convexity, performs better in volatile rate environments

In [ ]:
# Compare bullet vs barbell strategies
print("Bullet vs Barbell Strategy Comparison")
print("=" * 70)

# Bullet: $1M in 10Y bonds
bond_10y = Bond(face_value=1000, coupon_rate=0.05, maturity=10, frequency=2)
cash_flows_10y = bond_10y.get_cash_flows()
price_10y = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows_10y)
ytm_10y = bond_10y.yield_to_maturity(price_10y)

bullet_quantity = 1000000 / price_10y
bullet_value = 1000000
bullet_duration = bond_10y.duration(ytm_10y if ytm_10y else 0.05)
bullet_convexity = bond_10y.convexity(ytm_10y if ytm_10y else 0.05)

print("Bullet Strategy: 100% in 10Y bonds")
print(f"  Duration:  {bullet_duration:.2f}")
print(f"  Convexity: {bullet_convexity:.2f}")
print()

# Barbell: 50% in 2Y, 50% in 20Y (approximate same duration)
bond_2y = Bond(face_value=1000, coupon_rate=0.03, maturity=2, frequency=2)
bond_20y = Bond(face_value=1000, coupon_rate=0.055, maturity=20, frequency=2)

cash_flows_2y = bond_2y.get_cash_flows()
cash_flows_20y = bond_20y.get_cash_flows()

price_2y = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows_2y)
price_20y = sum(cf.amount * yc.get_discount_factor(cf.time) for cf in cash_flows_20y)

ytm_2y = bond_2y.yield_to_maturity(price_2y)
ytm_20y = bond_20y.yield_to_maturity(price_20y)

# Adjust weights to match duration
dur_2y = bond_2y.duration(ytm_2y if ytm_2y else 0.03)
dur_20y = bond_20y.duration(ytm_20y if ytm_20y else 0.055)

# Solve for weights: w * dur_2y + (1-w) * dur_20y = bullet_duration
w_2y = (dur_20y - bullet_duration) / (dur_20y - dur_2y) if dur_20y != dur_2y else 0.5
w_20y = 1 - w_2y

# Ensure positive weights
w_2y = max(0, min(1, w_2y))
w_20y = 1 - w_2y

barbell_duration = w_2y * dur_2y + w_20y * dur_20y
conv_2y = bond_2y.convexity(ytm_2y if ytm_2y else 0.03)
conv_20y = bond_20y.convexity(ytm_20y if ytm_20y else 0.055)
barbell_convexity = w_2y * conv_2y + w_20y * conv_20y

print("Barbell Strategy: Mix of 2Y and 20Y bonds")
print(f"  Weight in 2Y:  {w_2y*100:.1f}%")
print(f"  Weight in 20Y: {w_20y*100:.1f}%")
print(f"  Duration:      {barbell_duration:.2f}")
print(f"  Convexity:     {barbell_convexity:.2f}")
print()

print(f"Convexity advantage: {barbell_convexity - bullet_convexity:.2f}")
print()
print("Interpretation:")
print("The barbell has similar duration but higher convexity.")
print("This means it will outperform the bullet when rates move significantly")
print("(in either direction), but may underperform in stable markets.")

## 13. Summary and Key Takeaways

### Concepts Covered

1. **Yield Curve Basics**
   - Definition and importance
   - Par yields, spot rates, forward rates
   - Discount factors

2. **Interpolation Methods**
   - Cubic Spline: Smooth, exact fit
   - Nelson-Siegel: Parametric, economically intuitive
   - Nelson-Siegel-Svensson: More flexible

3. **Risk Metrics**
   - Duration: Price sensitivity to rate changes
   - Modified Duration: Practical approximation
   - Convexity: Second-order effects

4. **Curve Analysis**
   - Shape metrics: Level, Slope, Curvature
   - Scenario analysis: Parallel, steepening, flattening
   - Historical evolution

5. **Practical Applications**
   - Bond pricing with curves
   - Portfolio valuation and risk
   - Trading strategies: Riding, bullet vs barbell

### Using the FixedIncome Package

Key classes and methods:

```python
# Yield curve management
yc = YieldCurve(tenors, yields)
yc.get_yield(tenor)
yc.get_discount_factor(tenor)
yc.get_forward_rate(t1, t2)
yc.get_shape_metrics()

# Interpolation
interpolator = NelsonSiegelInterpolator()
interpolator.fit(tenors, yields)
yc.set_interpolator(interpolator)

# Bond pricing and risk
bond = Bond(face_value, coupon_rate, maturity, frequency)
bond.price(market_rate)
bond.duration(market_rate)
bond.convexity(market_rate)

# Data fetching
df = fetch_us_treasury_data(year)
curves = parse_treasury_data_to_yield_curves(df)
```

### Next Steps for Learning

1. **Bootstrapping**: Extract zero rates from par yields
2. **Cross-Currency**: Compare yield curves across countries
3. **Credit Spreads**: Extend to corporate bonds
4. **Advanced Models**: Vasicek, Hull-White, CIR for rates
5. **Portfolio Optimization**: Duration-neutral, curve-neutral strategies

### Further Reading

- Tuckman & Serrat: "Fixed Income Securities"
- Fabozzi: "Bond Markets, Analysis, and Strategies"
- Hull: "Options, Futures, and Other Derivatives" (Ch. 4-6)
- BIS Papers: "Zero-coupon yield curves: technical documentation"

## Appendix: Additional Exercises

To deepen your understanding, try these exercises:

1. **Interpolation Comparison**: Compare the three interpolation methods on yield curves from different dates. When does each perform best?

2. **Forward Rate Puzzle**: Calculate 1-year forward rates for each year from 1Y to 10Y. Plot them against spot rates. What does this tell you about market expectations?

3. **Duration Matching**: Create a portfolio of 2Y, 5Y, and 10Y bonds with a target duration of 7 years. What happens to its value under different scenarios?

4. **Roll-Down Analysis**: For each point on the curve, calculate the expected 1-year return assuming the curve stays constant. Where is the optimal maturity to invest?

5. **Historical Analysis**: Fetch yield curve data from previous years. Identify periods of inversion. Did recessions follow?

6. **Custom Strategy**: Design your own trading strategy based on curve shape. Backtest it using historical data.

Remember: The FixedIncome package provides the tools, but understanding the theory and practicing with real data is key to mastery.